In [5]:
import numpy as np
import random

def rank_links(distance_matrix):
    D = np.array(distance_matrix)
    
    # Get indices of upper triangle (k=1 excludes diagonal)
    i_indices, j_indices = np.triu_indices_from(D, k=1)
    
    # Extract corresponding distances
    distances = D[i_indices, j_indices]
    
    # Sort by distance
    sorted_idx = np.argsort(distances)
    
    # Build sorted list of (i, j, distance)
    ranked_links = [
        (int(i_indices[k]), int(j_indices[k]), float(distances[k]))
        for k in sorted_idx
    ]
    
    return ranked_links


def select_link(links,percentage_to_keep = 0.1, percentage_of_noise_added=0.1):
    #percentage of the total number of links to keep
    percentage_to_keep = 0.1
    #noise
    percentage_of_noise_added=0.1
    num_links_to_keep = int(len(links) * percentage_to_keep) 
    num_links_noise = int(len(links) * percentage_of_noise_added)   

    top_links = links[:num_links_to_keep]
    remaining_links = links[num_links_to_keep:]

    noise_links = random.sample(remaining_links, num_links_noise)

    # Combine
    final_links = top_links + noise_links
    return final_links

def write_links_to_file(links, filename):

    with open(filename, "w") as f:
        for i, j, d in links:
            f.write(f"2,{i},{j},{1.0}\n")


def load_coordinates(filename):
    data = np.loadtxt(filename, delimiter=",")
    
    node_ids = data[:, 0].astype(int)
    coords = data[:, 1:4]  # x, y, z
    
    return node_ids, coords


def compute_distance_matrix(coords):
    n = len(coords)
    dist_matrix = [[0.0]*n for _ in range(n)]
    
    for i in range(n):
        for j in range(n):
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            dz = coords[i][2] - coords[j][2]
            dist_matrix[i][j] = (dx**2 + dy**2 + dz**2) ** 0.5
            
    return dist_matrix

def compare_distance_maps(D1, D2):
    D1 = np.array(D1)
    D2 = np.array(D2)
    
    # Extract upper triangle
    i, j = np.triu_indices_from(D1, k=1)
    
    v1 = D1[i, j]
    v2 = D2[i, j]
    
    # Pearson correlation
    corr = np.corrcoef(v1, v2)[0, 1]
    
    return corr

In [2]:
from sequenceHandler import mapPDBToHMM

pfam_id = "PF00014"
#contanct map of the PDB and sequence
pdb_file = f"C:/Users/gfabi/Desktop/Internship/PDB/{pfam_id}.pdb"
chain_id = "A"
hmm_file = f"C:/Users/gfabi/Desktop/Internship/HMM/{pfam_id}.hmm"
hmm_file_2 = "None"        
output_map_file = "None"  

distance_map_pdb,map_index,aligned_sequence = mapPDBToHMM(pdb_file, chain_id, hmm_file, hmm_file_2, output_map_file, distType='all')

c:\Users\gfabi\miniconda3\envs\Internship\Lib\site-packages\Bio\PDB\Atom.py:237: PDBConstructionWarning: Used element 'U' for Atom (name=UNK) with given element 'X'
  warnings.warn(msg, PDBConstructionWarning)


STDERR: 


In [ ]:

scores=[]
for i in range(10):
    percentage_to_keep=i/10
    percentage_of_noise_added=0.0
    links = rank_links(distance_map_pdb)
    final_links = select_link(links, percentage_to_keep, percentage_of_noise_added)
    write_links_to_file(final_links, f"Links/top_{percentage_to_keep*100}%_links_{pfam_id}_noise_{percentage_of_noise_added*100}%.txt")

    #Magic

    filename = "GSE_output.txt" 
    node_ids, coords = load_coordinates(filename)
    distance_map_simul = compute_distance_matrix(coords)
    score=compare_distance_maps(distance_map_pdb, distance_map_simul)
    scores.append(score)

import matplotlib.pyplot as plt
plt.plot(range(10), scores)